# Phase 0 — Lead Management Data Foundry

Turns the raw **X Education `Leads.csv`** snapshot (9,240 rows, no timestamps) into a
MongoDB-ready, time-series lead management dataset.

**What this notebook does**
1. Cleans the `"Select"` sentinel, drops constant columns, canonicalises `Lead Source`
2. Synthesises `created_at` over ~14 months using a **marginal-preserving** assignment
3. Derives `converted_at` from channel-specific conversion lag
4. Generates sales reps, owner assignment, pipeline `stage`, an `activities` event log, and channel `spend`
5. Quarantines target-leakage columns under `analysis_only`
6. Validates invariants and exports MongoDB Extended JSON

**The key idea:** we never invent outcomes. Every count in the original file
(total leads per source, total conversions per source) is preserved exactly.
We only synthesise *when* things happened — by re-arranging real rows in time.

**Planted incidents.** The date assignment deliberately injects 5 known anomalies,
recorded in `_manifest.json`. That file is the **answer key** for your anomaly-detection
agent's eval suite — including one benign seasonal dip the agent should *not* flag.

In [ ]:
!pip -q install pandas numpy

import numpy as np, pandas as pd, json, os, zipfile
from datetime import timedelta

# ---------------- CONFIG ----------------
# Incident windows are short (8-26 days), so the realised effect sizes carry real
# sampling noise - across seeds the volume ratios centre on the planted values with
# sd ~0.08-0.20. Seed 7 was picked from a 12-seed sweep as the draw whose realised
# effects sit closest to the spec. Nothing is faked by this: the manifest records
# MEASURED effects either way, and any seed produces a valid dataset.
SEED         = 7
MONTHS_BACK  = 14      # >12 so month-over-year comparisons have both sides
LOCAL_BASE_D = 42      # +/- days either side of an incident used as its local baseline
OUT          = "out"

# The window always ENDS TODAY, so "yesterday" / "last week" / "this month" resolve
# no matter when this is run. Incidents are defined as offsets back from END
# (see Step 3), never as absolute dates - otherwise regenerating months later
# would silently push them outside the window and the anomalies would vanish.
END   = pd.Timestamp.now(tz="UTC").normalize()
START = (END - pd.DateOffset(months=MONTHS_BACK)).normalize()

rng  = np.random.default_rng(SEED)
DAYS = pd.date_range(START, END, freq="D")
D    = len(DAYS)

# START/END are midnights naming the first and last *day*. Timestamps land anywhere
# inside a day, so the real bound is the instant after END - treated as EXCLUSIVE,
# which guarantees every lead has a strictly positive amount of window left in which
# to convert (an inclusive bound admits room == 0 and a zero-length lag).
WINDOW_END = END + pd.Timedelta(days=1)

os.makedirs(OUT, exist_ok=True)
print(f"{D} days  |  {START.date()} -> {END.date()}")

In [ ]:
# ---------------- LOAD ----------------
try:
    from google.colab import files
    CSV = list(files.upload().keys())[0]
except Exception:
    CSV = "Leads.csv"

raw = pd.read_csv(CSV, encoding="utf-8-sig")
print("loaded:", raw.shape)
assert raw.shape[1] == 37, "unexpected column count - is this the right file?"

## Step 1 — Clean

Three separate problems, often conflated:

- **`"Select"`** is an unsubmitted dropdown, not a value. It affects City (24.3%),
  Specialization (21.0%) and "How did you hear" (54.6%). Leave it in and your agent
  will report *"Select"* as the top city.
- **Constant columns** carry zero information (one value across all 9,240 rows).
  They cost tokens in every schema prompt.
- **`Lead Source` casing/junk**: `Google` vs `google`, plus a literal `testone` record.
  Group before canonicalising and your channel report silently splits in two.

In [ ]:
df = raw.copy()

# "Select" and empty strings -> NaN; strip whitespace
def clean_cell(v):
    if isinstance(v, str):
        s = v.strip()
        return np.nan if s in ("", "Select") else s
    return v

df = df.apply(lambda s: s.map(clean_cell))

# drop constant columns
CONSTANT = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
print("dropping constants:", CONSTANT)
df = df.drop(columns=CONSTANT)

# canonicalise Lead Source
SOURCE_FIX = {
    "google": "Google", "bing": "Bing", "blog": "Blog",
    "welearnblog_Home": "WeLearn", "WeLearn": "WeLearn", "youtubechannel": "YouTube",
    "Press_Release": "Press Release", "Click2call": "Click2Call",
    "Pay per Click Ads": "Pay Per Click Ads", "testone": "__TEST__",
}
df["Lead Source"] = df["Lead Source"].replace(SOURCE_FIX)

n_test = int((df["Lead Source"] == "__TEST__").sum())
df = df[df["Lead Source"] != "__TEST__"].copy()
print(f"dropped {n_test} test record(s)")

# bucket the long tail so groupings stay readable
vc   = df["Lead Source"].value_counts()
RARE = set(vc[vc < 10].index)
df["Lead Source"] = df["Lead Source"].where(~df["Lead Source"].isin(RARE), "Other").fillna("Unknown")
print("\ncanonical sources:\n", df["Lead Source"].value_counts())

N = len(df)
ORIG_CONVERTED = int(df["Converted"].sum())
print(f"\nrows={N}  converted={ORIG_CONVERTED}  ({100*ORIG_CONVERTED/N:.1f}%)")

In [ ]:
# snake_case rename
RENAME = {
    "Prospect ID": "prospect_id", "Lead Number": "lead_number",
    "Lead Origin": "lead_origin", "Lead Source": "lead_source",
    "Do Not Email": "do_not_email", "Do Not Call": "do_not_call",
    "Converted": "converted", "TotalVisits": "total_visits",
    "Total Time Spent on Website": "time_on_site_sec",
    "Page Views Per Visit": "page_views_per_visit",
    "Last Activity": "last_activity", "Country": "country",
    "Specialization": "specialization",
    "How did you hear about X Education": "heard_from",
    "What is your current occupation": "occupation",
    "What matters most to you in choosing a course": "primary_motivation",
    "Search": "src_search", "Newspaper Article": "src_newspaper_article",
    "X Education Forums": "src_forums", "Newspaper": "src_newspaper",
    "Digital Advertisement": "src_digital_ad",
    "Through Recommendations": "src_recommendation",
    "Tags": "tags", "Lead Quality": "lead_quality", "Lead Profile": "lead_profile",
    "City": "city",
    "Asymmetrique Activity Index": "asym_activity_index",
    "Asymmetrique Profile Index": "asym_profile_index",
    "Asymmetrique Activity Score": "asym_activity_score",
    "Asymmetrique Profile Score": "asym_profile_score",
    "A free copy of Mastering The Interview": "wants_free_copy",
    "Last Notable Activity": "last_notable_activity",
}
df = df.rename(columns=RENAME)
df = df.reset_index(drop=True)

for c in ["do_not_email", "do_not_call", "wants_free_copy", "src_search",
          "src_newspaper_article", "src_forums", "src_newspaper",
          "src_digital_ad", "src_recommendation"]:
    if c in df: df[c] = (df[c] == "Yes")

df["total_visits"]         = df["total_visits"].fillna(0).astype(int)
df["time_on_site_sec"]     = df["time_on_site_sec"].fillna(0).astype(int)
df["page_views_per_visit"] = df["page_views_per_visit"].fillna(0.0).astype(float)
print(list(df.columns))

## Step 2 — Sales reps

Assigning owners *uniformly at random* produces a leaderboard where every rep converts
at 38.5% plus noise — useless for demoing "who needs coaching".

Instead each rep gets a `skill` multiplier, and assignment is **conditioned on the
outcome**: converted leads are drawn with probability proportional to `share x skill`,
non-converted to `share / skill`. Real spread, real totals, nothing fabricated.

In [ ]:
REP_NAMES = ["Aarti Deshpande","Rohit Malhotra","Sneha Iyer","Vikram Nair",
             "Priya Menon","Karan Shah","Divya Raghavan","Imran Qureshi",
             "Neha Kulkarni","Arjun Bose","Meera Pillai","Sahil Chopra"]
TEAMS = ["Inbound North","Inbound South","Enterprise"]

reps = []
for i, nm in enumerate(REP_NAMES):
    reps.append({
        "rep_id":   f"REP{i+1:03d}",
        "name":     nm,
        "team":     TEAMS[i % 3],
        "seniority": ["Junior","Mid","Senior"][min(i // 4, 2)],
        "skill":    float(np.round(rng.uniform(0.72, 1.34), 3)),
        "share":    float(np.round(rng.uniform(0.6, 1.5), 3)),
        "sla_discipline": float(np.round(rng.uniform(0.55, 1.45), 3)),
    })
reps_df = pd.DataFrame(reps)
display(reps_df)

## Step 3 — `created_at` (the core step)

A uniform random date gives you flat charts and nothing to discover. We build a daily
**intensity function** instead:

```
weight(day) = weekday x growth trend x annual seasonality x incident multipliers
```

Then, for every stratum `(lead_source, lead_origin, converted)`, we draw days from that
stratum's normalised weight vector. Because we only ever redistribute rows *within* a
stratum, every marginal in the original file survives untouched.

`vol_mult` shifts how many leads land in a window. `conv_mult` moves the conversion
*rate* — but naively applying it only to converted rows leaves a volume dent, so a
"conversion collapse" shows up as a volume drop too and the incident loses its point.

We therefore derive a compensating multiplier for the non-converted stratum. With
segment base rate `p`:

```
k = (1 - p * conv_mult) / (1 - p)
```

Expected volume then stays at exactly `vol_mult` while the rate moves as intended.
The same formula gives a **feasibility check**: if `p * conv_mult >= 1` the incident is
impossible and the cell raises rather than silently producing nothing. That guard is
why `reference_surge` is volume-only — Reference already converts at ~92%, so there is
no headroom for a lift, and `seo_content_revamp` carries the conversion-lift case
instead on a segment that has room.

In [ ]:
# Offsets are "days back from END", resolved to absolute dates below.
INCIDENTS = [
    dict(id="olark_conversion_collapse", t0=108, t1=88,
         kind="conversion_drop", segment={"lead_source": "Olark Chat"},
         vol_mult=1.00, conv_mult=0.32, should_alert=True,
         note="Chat widget mis-routed transcripts. Volume flat, conversion collapses."),
    dict(id="landing_page_outage", t0=227, t1=220,
         kind="volume_drop", segment={"lead_origin": "Landing Page Submission"},
         vol_mult=0.42, conv_mult=1.00, should_alert=True,
         note="JS error on the form; submissions fail silently."),
    dict(id="reference_surge", t0=318, t1=293,
         kind="volume_surge", segment={"lead_source": "Reference"},
         vol_mult=2.10, conv_mult=1.00, should_alert=True,
         note="Alumni referral push. Volume-only: Reference already converts ~92%, "
              "so there is no headroom for a conversion lift here."),
    dict(id="seo_content_revamp", t0=66, t1=50,
         kind="conversion_lift", segment={"lead_source": "Organic Search"},
         vol_mult=1.10, conv_mult=1.45, should_alert=True,
         note="Rewritten landing content. Conversion LIFT - the agent must flag "
              "improvements, not only regressions."),
    dict(id="festive_lull", t0=341, t1=332,
         kind="seasonal_benign", segment={},
         vol_mult=0.55, conv_mult=1.00, should_alert=False,
         note="Diwali. Expected every year - the agent should NOT raise an incident."),
    dict(id="google_spend_waste", t0=171, t1=153,
         kind="efficiency_drop", segment={"lead_source": "Google"},
         vol_mult=1.08, conv_mult=0.80, spend_mult=2.30, should_alert=True,
         note="Broad-match keyword expansion: spend 2.3x, yield flat. CAC spikes."),
]

for inc in INCIDENTS:
    inc["start"] = str((END - pd.Timedelta(days=inc["t0"])).date())
    inc["end"]   = str((END - pd.Timedelta(days=inc["t1"])).date())

dow      = DAYS.dayofweek.values
weekday  = np.where(dow >= 5, 0.40, 1.0) * np.where(dow == 0, 1.14, 1.0)
trend    = 0.78 + 0.46 * (np.arange(D) / D)
annual   = 1.0 + 0.16 * np.sin(2*np.pi*(DAYS.dayofyear.values/365.25) + 1.1)
BASE     = weekday * trend * annual

def window_mask(inc):
    return ((DAYS >= pd.Timestamp(inc["start"], tz="UTC")) &
            (DAYS <= pd.Timestamp(inc["end"],   tz="UTC"))).astype(bool)

WMASK = {inc["id"]: window_mask(inc) for inc in INCIDENTS}

def applies(inc, source, origin):
    seg = inc["segment"]
    if not seg: return True
    if "lead_source" in seg and seg["lead_source"] != source: return False
    if "lead_origin" in seg and seg["lead_origin"] != origin: return False
    return True

def seg_rows(seg):
    sub = df
    for k, v in seg.items(): sub = sub[sub[k] == v]
    return sub

# --- derive the non-converted multiplier so VOLUME stays flat -------------------
# Suppressing converted rows in a window without compensating leaves a volume dent,
# so a "conversion collapse" masquerades as a volume drop too. With segment base
# rate p, choosing k = (1 - p*conv_mult) / (1 - p) for the non-converted stratum
# keeps expected volume at exactly vol_mult while the rate moves as intended.
for inc in INCIDENTS:
    p, c = seg_rows(inc["segment"])["converted"].mean(), inc["conv_mult"]
    if p * c >= 1.0:
        raise ValueError(
            f"{inc['id']}: infeasible. base rate {p:.3f} x conv_mult {c} = {p*c:.3f} >= 1. "
            f"Max usable conv_mult here is {1/p:.2f} - this segment has no headroom.")
    inc["_nonconv_mult"] = (1 - p*c) / (1 - p)
    inc["_base_rate"]    = float(p)
    print(f"{inc['id']:28} base_rate={p:.3f}  conv_mult={c:.2f}  "
          f"-> nonconv_mult={inc['_nonconv_mult']:.3f}  "
          f"(expected rate {p*c/(p*c + (1-p)*inc['_nonconv_mult']):.3f})")

In [ ]:
created_idx = np.zeros(N, dtype=int)

for (source, origin, conv), grp in df.groupby(["lead_source","lead_origin","converted"]):
    w = BASE.copy()
    for inc in INCIDENTS:
        if not applies(inc, source, origin): continue
        m = WMASK[inc["id"]]
        w[m] *= inc["vol_mult"]
        w[m] *= inc["conv_mult"] if conv == 1 else inc["_nonconv_mult"]
    w = w / w.sum()
    created_idx[grp.index.values] = rng.choice(D, size=len(grp), p=w)

# intra-day: business-hours weighted (IST-ish)
HOUR_W = np.array([0.4,0.3,0.2,0.2,0.3,0.6,1.2,2.2,3.4,5.0,6.4,6.8,
                   5.6,4.4,5.0,6.0,6.4,5.8,4.6,3.6,2.8,2.0,1.2,0.7])
hours = rng.choice(24, size=N, p=HOUR_W/HOUR_W.sum())

df["created_at"] = (DAYS[created_idx]
                    + pd.to_timedelta(hours, unit="h")
                    + pd.to_timedelta(rng.integers(0, 3600, N), unit="s"))

print(df["created_at"].min(), "->", df["created_at"].max())
df.set_index("created_at").resample("ME").size().plot(
    kind="bar", figsize=(13,3), title="Leads created per month");

## Step 4 — `converted_at`

Conversion lag is right-skewed (most close fast, a few drag on), so lognormal is the
right family. Medians are channel-specific: a `Reference` closes in days, `Organic
Search` takes weeks.

Leads created near `END` would otherwise convert in the future. We resample the lag
into the remaining window rather than clamping, which avoids an artificial pile-up
on the final day.

In [ ]:
LAG_MEDIAN = {"Reference":3, "Welingak Website":4, "Olark Chat":9, "Facebook":12,
              "Google":14, "Referral Sites":15, "Direct Traffic":16,
              "Organic Search":21, "Bing":16, "Other":14, "Unknown":15}
SIGMA = 0.85

converted_at = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns, UTC]")
mask_conv = df["converted"] == 1
clamped = 0

for src, grp in df[mask_conv].groupby("lead_source"):
    med  = LAG_MEDIAN.get(src, 15)
    idx  = grp.index.values
    room = (WINDOW_END - grp["created_at"]).dt.total_seconds().values / 86400.0
    lag  = rng.lognormal(np.log(med), SIGMA, size=len(idx))
    for _ in range(25):                       # resample overshoots
        bad = lag > room
        if not bad.any(): break
        lag[bad] = rng.lognormal(np.log(med), SIGMA, size=int(bad.sum()))
    bad = lag > room
    clamped += int(bad.sum())
    if bad.any():
        # Fit the lag inside the remaining room. Never floor it to a fixed value:
        # a lead created minutes before the window closes has less room than any
        # floor you would pick, and the conversion would land outside the window.
        r = room[bad]
        lag[bad] = np.clip(r * rng.uniform(0.3, 0.95, int(bad.sum())), 1e-6, r * 0.999)
    converted_at.loc[idx] = grp["created_at"] + pd.to_timedelta(lag, unit="D")

df["converted_at"] = converted_at
df["days_to_convert"] = (df["converted_at"] - df["created_at"]).dt.total_seconds()/86400.0
print(f"clamped {clamped} lag(s) near window end")
print(df.groupby("lead_source")["days_to_convert"].median().sort_values().round(1))

## Step 5 — Owners and pipeline stage

`Converted` is binary, so out of the box there is no *open pipeline* — nothing to manage.
We derive an 8-state stage ladder from `Tags` (all 27 values mapped explicitly) so that
non-converted leads split into genuinely different situations: still working them,
can't reach them, or disqualified. "Lost" and "Unreachable" are very different problems.

In [ ]:
p_conv    = reps_df["share"] * reps_df["skill"];      p_conv    /= p_conv.sum()
p_nonconv = reps_df["share"] / reps_df["skill"];      p_nonconv /= p_nonconv.sum()

owner = np.empty(N, dtype=object)
ci = df.index[mask_conv].values
ni = df.index[~mask_conv].values
owner[ci] = rng.choice(reps_df["rep_id"].values, size=len(ci), p=p_conv.values)
owner[ni] = rng.choice(reps_df["rep_id"].values, size=len(ni), p=p_nonconv.values)
df["owner_id"] = owner

TAG_STAGE = {
    "Closed by Horizzon":"Won", "Lost to EINS":"Lost", "Lost to Others":"Lost",
    "Already a student":"Disqualified", "Not doing further education":"Disqualified",
    "Diploma holder (Not Eligible)":"Disqualified",
    "University not recognized":"Disqualified",
    "Recognition issue (DEC approval)":"Disqualified",
    "Interested in other courses":"Lost",
    "invalid number":"Unreachable", "wrong number given":"Unreachable",
    "number not provided":"Unreachable", "opp hangup":"Unreachable",
    "switched off":"Unreachable", "Ringing":"Attempting", "Busy":"Attempting",
    "Will revert after reading the email":"Engaged", "Still Thinking":"Engaged",
    "In confusion whether part time or DLP":"Engaged",
    "Want to take admission but has financial problems":"Qualified",
    "Interested in Next batch":"Qualified",
    "Shall take in the next coming month":"Qualified",
    "Interested  in full time MBA":"Qualified",
    "Graduation in progress":"Engaged", "Lateral student":"Engaged",
    "in touch with EINS":"Engaged",
}
ACT_STAGE = {
    "Email Bounced":"Unreachable", "Unreachable":"Unreachable",
    "Unsubscribed":"Disqualified", "Email Marked Spam":"Disqualified",
    "SMS Sent":"Attempting", "Email Opened":"Engaged",
    "Email Link Clicked":"Engaged", "Olark Chat Conversation":"Engaged",
    "Had a Phone Conversation":"Qualified", "Form Submitted on Website":"Engaged",
    "Converted to Lead":"Engaged", "Page Visited on Website":"New",
}

def stage_of(r):
    if r["converted"] == 1: return "Won"
    t = r["tags"]
    if isinstance(t, str) and t in TAG_STAGE:
        s = TAG_STAGE[t]
        return "Lost" if s == "Won" else s
    a = r["last_activity"]
    return ACT_STAGE.get(a, "New") if isinstance(a, str) else "New"

df["stage"] = df.apply(stage_of, axis=1)
OPEN = {"New","Attempting","Engaged","Qualified"}
df["is_open"] = df["stage"].isin(OPEN)
print(df["stage"].value_counts(), "\n\nopen pipeline:", int(df["is_open"].sum()))

## Step 6 — Activity event log

`last_activity` is an end-state with no path to it. We reconstruct a plausible journey:
an origin event at `created_at`, lead-side touches scaled by the **real** `total_visits`,
interleaved rep outreach, and the recorded `last_activity` as the final event.

`first_response_minutes` — time from creation to first rep touch — is what makes SLA
reporting possible, and it varies by rep via `sla_discipline`.

In [ ]:
ORIGIN_EVENT = {"API":"Chat Widget Opened", "Landing Page Submission":"Form Submitted on Website",
                "Lead Add Form":"Lead Added Manually", "Lead Import":"Lead Imported",
                "Quick Add Form":"Lead Added Manually"}
REP_EVENTS = ["Call Attempted","Email Sent","SMS Sent","Follow-up Call","Callback Scheduled"]
LEAD_EVENTS = ["Page Visited on Website","Email Opened","Email Link Clicked",
               "Olark Chat Conversation","Brochure Downloaded"]

sla_map = reps_df.set_index("rep_id")["sla_discipline"].to_dict()
activities, first_resp = [], np.full(N, np.nan)

for i, r in enumerate(df.itertuples(index=False)):
    t0  = r.created_at
    end = r.converted_at if pd.notna(r.converted_at) else t0 + timedelta(
              days=float(rng.uniform(1, 45)))
    if end <= t0: end = t0 + timedelta(hours=6)
    span = (end - t0).total_seconds()

    ev = [dict(lead_number=int(r.lead_number), ts=t0, actor="lead",
               type=ORIGIN_EVENT.get(r.lead_origin,"Lead Created"), channel=r.lead_source)]

    n_lead = int(np.clip(r.total_visits, 0, 12))
    for f in np.sort(rng.uniform(0.02, 0.95, n_lead)):
        ev.append(dict(lead_number=int(r.lead_number),
                       ts=t0 + timedelta(seconds=float(f*span)), actor="lead",
                       type=str(rng.choice(LEAD_EVENTS)), channel=r.lead_source))

    if r.stage != "New":
        delay = float(rng.lognormal(np.log(55*sla_map[r.owner_id]), 1.15))
        delay = min(delay, span/60)
        first_resp[i] = delay
        n_rep = int(np.clip(rng.poisson(2.4)+1, 1, 9))
        offs  = np.sort(np.concatenate([[delay*60],
                 rng.uniform(delay*60, max(span,delay*60+1), n_rep-1)]))
        for o in offs:
            ev.append(dict(lead_number=int(r.lead_number),
                           ts=t0 + timedelta(seconds=float(o)), actor="rep",
                           type=str(rng.choice(REP_EVENTS)), rep_id=r.owner_id))

    if isinstance(r.last_activity, str):
        ev.append(dict(lead_number=int(r.lead_number), ts=end, actor="lead",
                       type=r.last_activity, channel=r.lead_source, is_last=True))
    activities.extend(ev)

df["first_response_minutes"] = first_resp
df["sla_breached"] = df["first_response_minutes"] > 60

acts = pd.DataFrame(activities).sort_values(["lead_number","ts"]).reset_index(drop=True)
print(f"{len(acts):,} activities  |  {len(acts)/N:.1f} per lead")
print(f"SLA breach rate: {100*df['sla_breached'].mean():.1f}%")

## Step 7 — Channel spend

Without cost data you can report volume but never **efficiency** — and CPL / CAC / ROAS
are the questions a growth lead actually asks. Spend tracks daily lead volume with
noise, plus a floor so channels keep spending on quiet days.

Organic, Direct, Reference, Olark and Referral Sites are unpaid and get zero spend —
which is itself the finding: the best channels often cost nothing.

In [ ]:
PAID_CPL = {"Google":210.0, "Facebook":165.0, "Bing":190.0, "Other":240.0}
daily = (df.assign(day=df["created_at"].dt.floor("D"))
           .groupby(["day","lead_source"]).size().rename("leads").reset_index())

rows = []
for ch, target in PAID_CPL.items():
    sub = daily[daily["lead_source"] == ch].set_index("day")["leads"].reindex(DAYS, fill_value=0)
    noise = rng.normal(1.0, 0.16, D).clip(0.55, 1.8)
    spend = (sub.values + 1.6) * target * noise
    for inc in INCIDENTS:
        if "spend_mult" not in inc: continue
        if inc["segment"].get("lead_source") not in (None, ch): continue
        spend[WMASK[inc["id"]]] *= inc["spend_mult"]
    for d, lv, sp in zip(DAYS, sub.values, spend):
        rows.append(dict(day=d, channel=ch, spend_inr=round(float(sp), 2), leads=int(lv)))

spend_df = pd.DataFrame(rows)
print(f"total spend: INR {spend_df['spend_inr'].sum():,.0f}")
print(spend_df.groupby("channel")["spend_inr"].sum().round(0))

## Step 8 — Assemble documents & quarantine leakage

`tags`, `lead_quality` and the `asymmetrique_*` scores are assigned **after** an outcome
is known — `Tags` literally contains `"Closed by Horizzon"` for 358 won deals. They are
fine for descriptive BI, and poison for a predictive model (you'll see 98% accuracy and
ship something worthless).

Nesting them under `analysis_only` makes that boundary structural rather than a comment:
Phase 1's semantic layer can expose the subtree for reporting and exclude it from any
feature set by path.

In [ ]:
def iso(v):
    return None if pd.isna(v) else {"$date": pd.Timestamp(v).strftime("%Y-%m-%dT%H:%M:%S.000Z")}

def nn(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return None
    if isinstance(v, (np.integer,)): return int(v)
    if isinstance(v, (np.floating,)): return float(v)
    if isinstance(v, (np.bool_,)):    return bool(v)
    return v

lead_docs = []
for r in df.itertuples(index=False):
    lead_docs.append({
        "_id": r.prospect_id, "lead_number": int(r.lead_number),
        "created_at": iso(r.created_at), "converted_at": iso(r.converted_at),
        "converted": bool(r.converted), "days_to_convert": nn(r.days_to_convert),
        "stage": r.stage, "is_open": bool(r.is_open),
        "owner_id": r.owner_id,
        "lead_origin": nn(r.lead_origin), "lead_source": nn(r.lead_source),
        "engagement": {"total_visits": int(r.total_visits),
                       "time_on_site_sec": int(r.time_on_site_sec),
                       "page_views_per_visit": float(r.page_views_per_visit),
                       "last_activity": nn(r.last_activity)},
        "profile": {"country": nn(r.country), "city": nn(r.city),
                    "specialization": nn(r.specialization),
                    "occupation": nn(r.occupation),
                    "primary_motivation": nn(r.primary_motivation),
                    "heard_from": nn(r.heard_from)},
        "consent": {"do_not_email": bool(r.do_not_email),
                    "do_not_call": bool(r.do_not_call),
                    "wants_free_copy": bool(r.wants_free_copy)},
        "sla": {"first_response_minutes": nn(r.first_response_minutes),
                "breached": bool(r.sla_breached)},
        "analysis_only": {                       # <-- leakage. never a model feature.
            "tags": nn(r.tags), "lead_quality": nn(r.lead_quality),
            "lead_profile": nn(r.lead_profile),
            "asym_activity_index": nn(r.asym_activity_index),
            "asym_profile_index": nn(r.asym_profile_index),
            "asym_activity_score": nn(r.asym_activity_score),
            "asym_profile_score": nn(r.asym_profile_score),
            "last_notable_activity": nn(r.last_notable_activity)},
    })

act_docs = [{"lead_number": int(a.lead_number), "ts": iso(a.ts), "actor": a.actor,
             "type": a.type, "channel": nn(getattr(a, "channel", None)),
             "rep_id": nn(getattr(a, "rep_id", None)),
             "is_last": bool(getattr(a, "is_last", False) is True)}
            for a in acts.itertuples(index=False)]

spend_docs = [{"day": iso(s.day), "channel": s.channel,
               "spend_inr": float(s.spend_inr), "leads": int(s.leads)}
              for s in spend_df.itertuples(index=False)]

rep_docs = [{"_id": r["rep_id"], **{k: v for k, v in r.items() if k != "rep_id"}}
            for r in reps_df.to_dict("records")]
print(len(lead_docs), len(act_docs), len(spend_docs), len(rep_docs))

## Step 9 — Validate

A generator you can't verify is a liability. These assertions are the contract Phase 1
depends on; if any fail, do not export.

In [ ]:
errs = []
def check(cond, msg):
    (print("  ok  " + msg) if cond else errs.append(msg))

check(len(lead_docs) == N, f"lead count == {N}")
check(sum(d["converted"] for d in lead_docs) == ORIG_CONVERTED,
      f"conversions preserved == {ORIG_CONVERTED}")
check(len({d["_id"] for d in lead_docs}) == N, "prospect_id unique")
check(all(d["converted_at"] is None for d in lead_docs if not d["converted"]),
      "no converted_at on unconverted leads")
check(all(d["converted_at"] is not None for d in lead_docs if d["converted"]),
      "every converted lead has converted_at")
check(df.loc[mask_conv, "days_to_convert"].min() > 0, "converted_at strictly after created_at")
check(df["created_at"].min() >= START and df["created_at"].max() < WINDOW_END,
      "created_at in window")
check(df.loc[mask_conv, "converted_at"].max() < WINDOW_END, "converted_at within window")
check(set(df["owner_id"]) <= set(reps_df["rep_id"]), "every owner_id resolves to a rep")
check(not df["stage"].isna().any(), "stage assigned for every lead")
check((df.loc[df["converted"]==1, "stage"] == "Won").all(), "converted <=> Won")

# the original marginals must survive the time assignment
orig_src = raw["Lead Source"].replace(SOURCE_FIX)
orig_src = orig_src[orig_src != "__TEST__"]
orig_src = orig_src.where(~orig_src.isin(RARE), "Other").fillna("Unknown")
check(df["lead_source"].value_counts().equals(orig_src.value_counts()),
      "lead_source marginals preserved exactly")

print("\n" + ("ALL CHECKS PASSED" if not errs else "FAILED:\n" + "\n".join(errs)))
assert not errs

In [ ]:
# ---- measure what actually landed in the data -------------------------------
# Compare each window against a LOCAL baseline (+/- LOCAL_BASE_D days, with every
# other incident window removed) rather than the all-time mean. The all-time mean
# carries the growth trend and annual seasonality, which biases short windows and
# makes a correct incident look mis-sized.
ANY_INC = np.zeros(D, bool)
for i_ in INCIDENTS: ANY_INC |= WMASK[i_["id"]]

day_idx_all = (df["created_at"].dt.normalize() - START).dt.days.values

def local_baseline_mask(inc):
    idx = np.where(WMASK[inc["id"]])[0]
    lo, hi = idx.min(), idx.max()
    b = np.zeros(D, bool)
    b[max(0, lo-LOCAL_BASE_D):lo]            = True
    b[hi+1:min(D, hi+1+LOCAL_BASE_D)]        = True
    return b & ~ANY_INC

def measure(inc):
    sub  = seg_rows(inc["segment"])
    di   = (sub["created_at"].dt.normalize() - START).dt.days.values
    wm, bm = WMASK[inc["id"]], local_baseline_mask(inc)
    inw, inb = wm[di], bm[di]
    conv = sub["converted"].values.astype(float)
    vw, vb = inw.sum()/wm.sum(), inb.sum()/bm.sum()
    cw = conv[inw].mean() if inw.sum() else float("nan")
    cb = conv[inb].mean() if inb.sum() else float("nan")
    m = {"window_days": int(wm.sum()), "baseline_days": int(bm.sum()),
         "leads_in_window": int(inw.sum()),
         "volume_per_day": {"baseline": round(vb,3), "window": round(vw,3),
                            "ratio": round(vw/vb,3)},
         "conversion_rate": {"baseline": round(cb,4), "window": round(cw,4),
                             "ratio": round(cw/cb,3)}}
    if "spend_mult" in inc:
        ch = inc["segment"].get("lead_source")
        s  = spend_df[spend_df["channel"] == ch].set_index("day")["spend_inr"]
        s  = s.reindex(DAYS, fill_value=0.0).values
        sw, sb = s[wm].mean(), s[bm].mean()
        cac_w = s[wm].sum()/max(conv[inw].sum(), 1)
        cac_b = s[bm].sum()/max(conv[inb].sum(), 1)
        m["spend_per_day"] = {"baseline": round(sb,1), "window": round(sw,1),
                              "ratio": round(sw/sb,3)}
        m["cac_inr"] = {"baseline": round(cac_b,1), "window": round(cac_w,1),
                        "ratio": round(cac_w/cac_b,3)}
    return m

print(f"{'incident':28} {'vol planted->measured':>24} {'conv planted->measured':>24}")
print("-"*80)
for inc in INCIDENTS:
    inc["measured"] = measure(inc)
    v, c = inc["measured"]["volume_per_day"], inc["measured"]["conversion_rate"]
    print(f"{inc['id']:28} {inc['vol_mult']:>10.2f} -> {v['ratio']:<10.2f} "
          f"{inc['conv_mult']:>10.2f} -> {c['ratio']:<10.2f}")

TOL = 0.22
drift = [(i["id"], k, p, m) for i in INCIDENTS
         for k, p, m in [("vol",  i["vol_mult"],  i["measured"]["volume_per_day"]["ratio"]),
                         ("conv", i["conv_mult"], i["measured"]["conversion_rate"]["ratio"])]
         if abs(m - p)/p > TOL]
print("\n" + ("all effects within +/-22% of planted" if not drift else
      "DRIFT (small-sample windows can exceed tolerance legitimately):\n" +
      "\n".join(f"  {a} {b}: planted {c:.2f} measured {d:.2f}" for a,b,c,d in drift)))

## Step 10 — Export

MongoDB **Extended JSON** (`{"$date": ...}`) so `mongoimport` creates real BSON dates.
Import them as strings and every `$dateTrunc` / range query in Phase 1 breaks.

`_manifest.json` carries the seed, the window, row counts **and the planted incidents** —
that last block is the ground truth your anomaly-detection evals assert against.

In [ ]:
def write_jsonl(name, docs):
    p = os.path.join(OUT, name)
    with open(p, "w", encoding="utf-8") as f:
        for d in docs: f.write(json.dumps(d, ensure_ascii=False) + "\n")
    print(f"{name:24} {len(docs):>7,} docs  {os.path.getsize(p)/1e6:.1f} MB")

write_jsonl("leads.jsonl",   lead_docs)
write_jsonl("activities.jsonl", act_docs)
write_jsonl("channel_spend.jsonl", spend_docs)
write_jsonl("reps.jsonl",    rep_docs)

manifest = {
    "generated_at": pd.Timestamp.utcnow().isoformat(),
    "seed": SEED, "source_file": CSV,
    "window": {"start": str(START.date()), "end": str(END.date())},
    "counts": {"leads": len(lead_docs), "activities": len(act_docs),
               "channel_spend": len(spend_docs), "reps": len(rep_docs),
               "converted": ORIG_CONVERTED},
    "synthetic_fields": ["created_at","converted_at","days_to_convert","owner_id",
                         "stage","is_open","sla.*","activities.*","channel_spend.*","reps.*"],
    "authentic_fields": ["converted","lead_origin","lead_source","engagement.*",
                         "profile.*","consent.*","analysis_only.*"],
    "leakage_quarantine": "analysis_only.* is post-hoc; exclude from any model feature set",
    "dropped_constant_columns": CONSTANT,
    # ---- eval ground truth -------------------------------------------------
    # "planted" is what we asked for; "measured" is what the exported data
    # actually contains. Assert evals against MEASURED. They differ whenever a
    # segment saturates or a window is small, and a stale answer key will have
    # you debugging a correct agent.
    "incidents": [{
        "id": i["id"], "kind": i["kind"], "segment": i["segment"],
        "start": i["start"], "end": i["end"],
        "days_ago": {"start": i["t0"], "end": i["t1"]},
        "should_alert": i["should_alert"], "note": i["note"],
        "planted": {"vol_mult": i["vol_mult"], "conv_mult": i["conv_mult"],
                    **({"spend_mult": i["spend_mult"]} if "spend_mult" in i else {}),
                    "segment_base_rate": round(i["_base_rate"], 4),
                    "derived_nonconv_mult": round(i["_nonconv_mult"], 4)},
        "measured": i["measured"],
    } for i in INCIDENTS],
}
with open(os.path.join(OUT,"_manifest.json"),"w") as f:
    json.dump(manifest, f, indent=2, default=str)

with zipfile.ZipFile("lead_foundry_output.zip","w",zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(OUT): z.write(os.path.join(OUT,fn), fn)

try:
    from google.colab import files as _f; _f.download("lead_foundry_output.zip")
except Exception:
    print("\nwrote lead_foundry_output.zip")

## Import into MongoDB

```bash
unzip lead_foundry_output.zip -d foundry && cd foundry
for c in leads activities channel_spend reps; do
  mongoimport --uri "$MONGODB_URI" --db leadops --collection $c \
              --file $c.jsonl --type json --drop
done
```

Then create the indexes Phase 1's query compiler will assume exist:

```js
db.leads.createIndex({ created_at: 1 })
db.leads.createIndex({ lead_source: 1, created_at: 1 })
db.leads.createIndex({ owner_id: 1, created_at: 1 })
db.leads.createIndex({ stage: 1, is_open: 1 })
db.leads.createIndex({ converted: 1, converted_at: 1 })
db.activities.createIndex({ lead_number: 1, ts: 1 })
db.activities.createIndex({ ts: 1, actor: 1 })
db.channel_spend.createIndex({ day: 1, channel: 1 }, { unique: true })
```

**Sanity query** — weekly conversion rate for Olark Chat. If `olark_conversion_collapse`
shows up in June 2026, your pipeline is end-to-end correct:

```js
db.leads.aggregate([
  { $match: { lead_source: "Olark Chat" } },
  { $group: {
      _id: { $dateTrunc: { date: "$created_at", unit: "week" } },
      leads: { $sum: 1 }, won: { $sum: { $cond: ["$converted", 1, 0] } } } },
  { $project: { leads: 1, won: 1,
      rate: { $round: [{ $multiply: [{ $divide: ["$won", "$leads"] }, 100] }, 1] } } },
  { $sort: { _id: 1 } }
])
```

Ping me once this is loaded and I'll build the Phase 1 semantic layer against this exact schema.